# Libraries

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import seaborn as sns
from pathlib import Path
from arch import arch_model


# Pobieramy ścieżkę do folderu wyżej (czyli głównego folderu projektu)
project_root = os.path.abspath('..')

# Dodajemy ją do listy ścieżek, w których Python szuka modułów
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.load_data import load_raw_djia
from src.data.preprocess import sort_data_by_date, calculate_returns, add_garch_residuals, normalize_to_uniform

print(f"Dodano ścieżkę główną: {project_root}")

Dodano ścieżkę główną: /mnt/FC3E32CE3E3281A6/Users/Stanislav/github_desktop_wd/mamba-joint-distribution-finance


# Loading data

In [13]:
path1 = '../data/raw/djia29/AAPL.csv'
path2 = '../data/raw/djia29/AXP.csv'

with open(path1, 'r') as f:
    data_raw1 = pd.read_csv(f, index_col='date', parse_dates=['date'], skiprows = [1])

with open(path2, 'r') as f:
    data_raw2 = pd.read_csv(f, index_col='date', parse_dates=['date'], skiprows = [1])

In [14]:
data1 = data_raw1.copy()
data2 = data_raw2.copy()

data1 = data1.astype(float)
data1 = data1.sort_index(ascending=True)

data2 = data2.astype(float)
data2 = data2.sort_index(ascending=True)

In [15]:
data1['returns'] = data1['close'].pct_change()
data1['log_returns'] = np.log(data1['close'] / data1['close'].shift(1))

data2['returns'] = data2['close'].pct_change()
data2['log_returns'] = np.log(data2['close'] / data2['close'].shift(1))
data1.head()

,close,volume,open,high,low,returns,log_returns
date,,,,,,,
2008-08-14,25.6171,177684343.0,25.4757,25.7786,25.4057,NaN,NaN
2008-08-15,25.1057,176996663.0,25.5771,25.6786,25.0071,-0.019963,-0.020165
2008-08-18,25.0557,137832351.0,25.0814,25.4014,24.8314,-0.001992,-0.001994
2008-08-19,24.7900,153915766.0,24.9343,25.2957,24.5443,-0.010604,-0.010661
2008-08-20,25.1200,126718798.0,24.9676,25.2771,24.8014,0.013312,0.013224


# Tryout of HCR on AAPL and AXP

In [16]:
from scipy.special import eval_legendre
import numpy as np
import pandas as pd

def calculate_hcr_coefficients(df: pd.DataFrame, column: str = 'uniform_returns', max_degree: int = 4) -> dict:
    # 1. Wyciągamy nasze płaskie dane [0, 1]
    x_uniform = df[column].dropna().values
    
    # 2. TWÓJ POMYSŁ: Kompresujemy (a właściwie rozciągamy) do [-1, 1]
    x_scaled = 2 * x_uniform - 1
    
    coefficients = {}
    
    for i in range(1, max_degree + 1):
        # 3. Używamy ZWYKŁEGO wielomianu Legendre'a na przeskalowanych danych
        # Pierwiastek normalizujący sqrt(2i + 1) zostaje bez zmian!
        normalization_factor = np.sqrt(2 * i + 1)
        
        poly_values = eval_legendre(i, x_scaled) * normalization_factor
        
        # 4. Wyliczamy średnią
        a_i = np.mean(poly_values)
        coefficients[f'a_{i}'] = a_i
        
    return coefficients

In [18]:
add_garch_residuals(data1, column='log_returns')
add_garch_residuals(data2, column='log_returns')

,close,volume,open,high,low,returns,log_returns,garch_residuals
date,,,,,,,,
2008-08-14,38.20,11987080.0,36.66,38.40,36.51,NaN,NaN,NaN
2008-08-15,39.07,15295070.0,38.60,39.71,38.46,0.022775,0.022519,0.447015
2008-08-18,37.99,12374440.0,39.20,39.34,37.58,-0.027643,-0.028032,-0.622796
2008-08-19,36.73,13363730.0,37.29,37.29,36.18,-0.033167,-0.033729,-0.768268
2008-08-20,37.43,13003770.0,36.83,38.25,36.36,0.019058,0.018879,0.406845
...,...,...,...,...,...,...,...,...
2018-08-08,102.78,2228875.0,102.00,102.95,101.81,0.007944,0.007912,0.551648
2018-08-09,102.99,2434972.0,102.44,103.32,102.28,0.002043,0.002041,0.090001
2018-08-10,101.58,2473161.0,102.09,102.20,101.17,-0.013691,-0.013785,-1.223664
